# SQLAlchemy: jointures, requêtes complexes et opérations CRUD

On a vu dans le cours précédent:
- comment faire des requêtes SQL (textuelles et ORM)
- comment décrire des tables individuelles via des `db.Models` SQLAlchemy
- comment faire des requêtes `SELECT` sur une table

Aujourd'hui, on va voir: 
- **les jointures SQL** et tables de relation
- **comment utiliser les jointures en SQLAlchemy**
- le reste des requêtes CRUD: **read**, **update** et **delete**


In [ ]:
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)
print(db)


--- 

# Modéliser des jointures

Si on reprend le dernier cours, on a modélisé deux tables: `Iconography` et `Author`.

```py
# comment lit-on chacun des attributs ci-dessous ?
class Iconography(db.Model):
    __tablename__ = "Iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    # NOTE: il reste à définir la relation avec la table `author`, met on va le faire aujourd'hui !
    id_author = ...


class Author(db.Model):
    __tablename__ = "author"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    author_name: Mapped[str] = mapped_column(unique=True)
```

Si on reprend notre modèle de base de données ci-dessous, on voit plus globalement qu'on a deux types de relations:
- **one to many**: `Author <-> Iconography`
- **many to many**: `Iconography <-> Place` et `Iconography <-> Theme`
- (et pour rappel, les relations *many to one*, c'est la même chose que des relations *one to many* mais à l'envers)

![db schema](./img/db_schema.png)

## One to many: `Author <-> Iconography`

`author` a une relation one to many à `iconography`:
- 1 ressource iconographique a un.e seul.e auteur.ice, 
- mais une entrée de la table `author` peut être associée à plusieurs ressources iconographiques.

**Pour rappel, en SQL, cela est modélisé par une clé étrangère sur la table `iconography`** qui pointe vers `author`: `iconography.id_author`.

**Avec SQLAlchemy**, pour décrire une relation one-to-many, on doit définir:
- **la colonne `Iconography.id_author`**, une `ForeignKey` vers `author.id`.
- **la propriété `Iconography.author`**, qui permettra d'accéder à l'auteur.ice d'une ressource icono
- **la propriété `Author.iconography`**, qui permettra d'accéder aux ressources iconographiques associées à un.e auteur.ice.

```py
class Author(db.Model):
    __tablename__ = "author"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    author_name: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="author"
    )

class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    id_author: Mapped[Optional[int]] = mapped_column(ForeignKey("author.id"))    

    author: Mapped[Optional["Author"]] = relationship(
        back_populates="iconography"
    )
```

### Modéliser une clé étrangère: `Iconography.id_author`

Voici comment on définit une clé étrangère:

```py
class Iconography(db.Model):
    id_author: Mapped[Optional[int]] = mapped_column(ForeignKey("author.id"))
```

On définit une clé étrangère presque comme n'importe quelle colonne.

#### `Mapped[Optional[int]]`: typer une clé étrangère

- `Mapped` indique qu'on a à faire à une colonne
- `Optional` indique que le champ `Iconography.id_author` peut être nullable (l'auteurice d'une ressource iconographique n'a pas besoin d'être défini.e)
- `int`, c'est le type de la clé étrangère
- **en résumé**: `Iconography.id_author` est une colonne stockant des entiers nullables.

#### `mapped_column(ForeignKey("author.id"))`: définir une clé étrangère

- on définit les contraintes supplémentaires de la colonne avec `mapped_column()` 
- **la particularité est: `ForeignKey("author.id")`**: on définit le contenu de la colonne comme une clé étrangère qui pointe vers la colonne `author.id`.

### Modéliser les `relationships`: `Iconography.author` et `Author.iconography`

Contrairement à `iconography.id_author`, ces propriétés contiennent du nouveau:

```py
class Iconography(db.Model):
    author: Mapped[Optional["Author"]] = relationship(
        back_populates="iconography"
    )

class Author(db.Model):
    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="author"
    )
```

`Iconography.author` et `Author.iconography` **permettront d'accéder aux objets liés**: 
- l'objet `Author` associé à une ressource iconographique se trouve dans `Iconography.author` 
- `Author.iconography` contient la liste d'objets `Iconography` associés à un `Author`
- **à noter**: `Iconography.author` et `Author.iconography` **ne sont pas des colonnes**: c'est des propriétés propres à SQLAlchemy et qui permettent de **ne pas avoir à faire de jointures à la main**

**Pour la syntaxe**, prenons:

```py
# dans la classe `Iconography`
author: Mapped[Optional["Author"]] = relationship(
    back_populates="iconography"
)
```

####  `Mapped[Optional["Author"]]`: la définition du type

`Mapped[Optional["Author"]]` indique que `Iconography` est **associé à un 0 ou 1 objet `Author`**:

- `"Author"`, c'est **le nom de notre modèle SQLAlchemy** pour la table `author`
- `Optional` permet d'indiquer que il y à une relation de 0 à *n* entre `Iconography` et `Author`. 
    - l'usage de `Optional` est aligné avec le type de `Iconography.id_author`

#### `relationship(back_populates="iconography")`: la relation entre les tables

- la fonction `relationship()` permet de définir ce champ comme une **relation SQLAlchemy** (renvoi vers les objets d'un autre modèle)
- on utilise `back_populates="iconography"` pour **lier la propriété `Author.iconography` à `Iconography.author`** (qu'on a aussi défini). Cela permet à SQLAlchemy de synchroniser les valeurs de ces deux propriétés.

#### Un autre exemple:

À partir de là, comment interpréter `iconography` ci-dessous ?

```py
# dans la classe `Author`
iconography: Mapped[List["Iconography"]] = relationship(
    back_populates="author"
)
```

## Many-to-many: `Iconography <-> Place`

La manière de faire des relations many-to-many est très semblable, sauf que l'on doit aussi définir une *table secondaire*: la table de relation.

**On va modéliser la relation entre `Iconography` et `Place`**.

**En SQLAlchemy, pour modéliser une relation one-to-many**, il faut:
- définir `IconographyPlace`, la table de relation entre `Iconography` et `Place`
- définir `Iconography.place`, la propriété permettant d'accéder aux objets `Place` depuis `Iconography`
- définir `Place.iconography.`, la propriété permettant d'accéder aux objets `Iconography` depuis `Place` (inverse de `Iconography.place`, donc).

### La table secondaire: `IconographyPlace`

Voici notre modèle pour `IconographyPlace`, table de relation entre `Iconography` et `Place`. Rien de bien surprenant ici.

```py
class IconographyPlace(db.Model):
    __tablename__ = "iconography_place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_place: Mapped[int] = mapped_column(ForeignKey("place.id"))
```

### Définir la relation entre `Iconography` et `Place`: `Iconography.place` et `Place.iconography`

En SQLAlchemy, on définit une relation many to many **directement entre les tables `Iconography` et `Place`**. `IconographyPlace` est utilisé implicitement par SQLAlchemy.

#### Définir `Iconography.place`

```py
class Iconography(db.Model):
    __tablename__ = "iconography"
    # ...
    place: Mapped[List["Place"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="iconography"
    )
```

- l'attribut `place` permet d'accéder aux objets `Place` depuis le modèle `Iconography` 
- avec `Mapped[List["Place"]]`, on explique que `Iconography.place` est une liste d'objets `Place`
- comme dit plus haut, `relationship()` est la fonction qui permet de créer une relation entre deux Models. **Ici, sa syntaxe est particulière**.

#### `relationship()` pour les relations many to many

```py
relationship(
    secondary=IconographyPlace.__table__,
    back_populates="iconography"
)
```

- `back_populates="iconography"` permet d'associer le champ `Iconography.place` à `Place.iconography`.
- **la nouvelle syntaxe, c'est: `secondary=IconographyPlace.__table__`**: 
    - `secondary` permet de définir une **une table de relation entre `Iconography` et `Place`**: `IconographyPlace`
    - cela veut dire que le lien `Iconography <-> IconographyPlace <-> Place` n'a pas besoin d'être défini: `IconographyPlace` est implicitement géré par SQLAlchemy.
    - on note que on utilise `IconographyPlace.__table__`, pas juste `IconographyPlace`

#### Configurer la relation inverse: `Place`

**`Place` est défini de la même manière**:

```py
class Place(db.Model):
    __tablename__ = "place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    address: Mapped[Optional[str]]
    richelieu_url: Mapped[str] = mapped_column(unique=True)
    # JSON est un type SQLAlchemy
    loc: Mapped[Dict] = mapped_column(JSON)
    plot: Mapped[Dict] = mapped_column(JSON)
    date_lower: Mapped[int]
    date_upper: Mapped[int]

    # comment interpétez vous cette propriété ?
    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="place"
    )
```

### Exercice: modéliser la relation many-to-many entre `Iconography` et `Theme`

**Complétez le modèle de données ci-dessous**:
- modélisez la table `IconographyTheme`
- modélisez la table `Theme`, y compris la relation entre `Theme` et `Iconography`
- complétez `Iconography` pour faire le lien avec `Theme`.

In [ ]:
from typing import List, Dict, Optional

from sqlalchemy import JSON, ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship, Mapped


APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)

# IconographiePlace est bien définie comme il faut
class IconographyPlace(db.Model):
    __tablename__ = "iconography_place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_place: Mapped[int] = mapped_column(ForeignKey("place.id"))

# la classe Place est bien définie comme il faut
class Place(db.Model):
    __tablename__ = "place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    address: Mapped[Optional[str]]
    richelieu_url: Mapped[str] = mapped_column(unique=True)
    # JSON est un type SQLAlchemy
    loc: Mapped[Dict] = mapped_column(JSON)
    plot: Mapped[Dict] = mapped_column(JSON)
    date_lower: Mapped[int]
    date_upper: Mapped[int]

    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="place"
    )

# Author est bien définie comme il faut
class Author(db.Model):
    __tablename__ = "author"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    author_name: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="author", 
    )

# à vous de jouer ! ---------------------------------------------------------

# 1. définir `IconographyTheme`, le modèle de la table de relation entre `Iconography` et `Theme`
# voir `IconographiePlace` pour un exemple

# 2. définir `Theme` et sa relation avec `Iconography` . 
# Voir:
#   - le modèle de `theme` dans le diagramme au début du cours. 
#   - voir `Place` pour un exemple de relation avec `Iconography`

# 3. complétez Iconography pour faire le lien avec `Theme`.
class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    id_author: Mapped[Optional[int]] = mapped_column(ForeignKey("author.id"))
    
    author: Mapped[Optional["Author"]] = relationship(
        back_populates="iconography"
    )
    place: Mapped[List["Place"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="iconography"
    )




---

# Requêtes `SELECT` sur des `relationships`

## Accéder aux relations

On l'a vu, pour chaque table qui a une clé étrangère, on a définit dans notre modèle des propriétes `relationship()`. Voilà comment on les utilise:

In [ ]:
with app.app_context():
    icono_list = db.session.execute(db.select(Iconography)).scalars().all()
    for icono_item in icono_list:
        print(f"Icono #{icono_item.id} ****************")
        # comment interpréter ce qui s'affiche ?
        print(icono_item.author)
        if icono_item.author is not None:
            # et ici ?
            print(icono_item.author.author_name)
        else: 
            print("Pas d'auteur.ice pour cette ressoyurce iconographique")

On voit donc que **`relationship() permet d'accéder aux objets `Author` directement depuis `Iconography`**.

Et inversement:

In [ ]:
with app.app_context():
    author_list = db.session.execute(db.select(Author)).scalars().all()
    for author_item in author_list:
        print(f"Auteurice #{author_item.id}: {author_item.author_name} *************")
        print(f"{len(author_item.iconography)} ressources iconographiques liées.")
        print(author_item.iconography)

## Filtrer sur des relations

### `.has()`: filtrer sur une relation *many to one*

Pour rappel, `Iconography.author` contient 0 ou 1 objets `Author`.

La requête ci-dessous permet d'afficher **toutes les ressources iconographiques qui ont pour auteur Eugène Atget**.

En SQL, on écrirait:

```sql
SELECT * FROM iconography
JOIN author ON iconography.id_author = author.id
WHERE author.author_name = 'Atget Eugène, photographe';
```

Avec SQLAlchemy, on écrit la requête `query` suivante.

In [ ]:
with app.app_context():
    query = db.select(Iconography).filter(
        Iconography.author.has(Author.author_name == "Atget Eugène, photographe")
    )
    results = db.session.execute(query).scalars().all()
    print(f"il y a {len(results)} photograhpies de Atget Eugène, photographe")
    print(results)

**Décomposons** la syntaxe:

```py
query = db.select(Iconography).filter(
    Iconography.author.has(Author.author_name == "Atget Eugène, photographe")
)
```

Prenons **ce qu'on connaît**:

- `db.select(Iconography)` : on demande les lignes de la table `iconography
- `.filter()` : on ajoute une condition pour ne garder que certaines lignes de `iconography
- `Author.author_name == "Atget Eugène, photographe"` : une condition simple

**Ce qui est nouveau, c'est `.has()`**:
- `.has()` permet de **filtrer une relation**
- notre requête correspond à: **"Garde les `Iconography` dont l'`author` associé satisfait la condition:** `Author.author_name == "Atget Eugène, photographe"`. 

En résumé, **`.has()` qui permet de filtrer les lignes d'une table par les propriétés d'une autre, quand la relation ne contient que 1 objet**.

### `.any()`: filtrer sur une relation *one-to-many*

Quand une relation contient une liste d'objets (comme pour `Author.iconography`), on utilise **`.any()` à la place de `.has()`**. La syntaxe est très similaire:

La requête ci-dessous permet d'afficher **tous les auteurs qui ont au moins une ressource iconographique datée de 1900 ou après**.

En SQL, on écrirait:

```sql
SELECT * FROM author
JOIN iconography ON author.id = iconography.id_author
WHERE iconography.date_lower >= 1900;
```

Avec SQLAlchemy, on écrit la requête `query` suivante:

In [ ]:
with app.app_context():
    query = db.select(Author).filter(
        Author.iconography.any(Iconography.date_lower >= 1900)
    )
    results = db.session.execute(query).scalars().all()
    print(results)


**La syntaxe est la même que pour `.has()`**:

```py
query = db.select(Author).filter(
    Author.iconography.any(Iconography.date_lower == 1900)
)
```

- `db.select(Author)` : on demande les lignes de la table `author`
- `.filter()` : on ajoute une condition pour ne garder que certaines lignes de `author`
- `Iconography.date_lower == 1900` : une condition simple
- `Author.iconography.any()` permet de **filtrer une relation**: `Author` par `Iconography`.

Notre requête correspond à: **"Garde les `Author` dont au moins une `Iconography` associée satisfait la condition:** `Iconography.date_lower == 1900`". 

En résumé, **`.any()` permet de filtrer les lignes d'une table par les propriétés d'une autre, quand la relation contient une liste d'objets**.

### Résumé

#### `.has()` vs `.any()`: tableau comparatif

| aspect | `.has()` | `.any()` |
|--------|----------|---------|
| **type de relation** | many-to-one | one-to-many/many-to-many |
| **exemple** | ressources icono d'un auteur spécifique | auteurs ayant des iconographies datées d'après 1900 |
| **retour** | objet unique ou vide | zéro, un ou plusieurs objets |

On peut bien sûr combiner les filtres avec `and_()`, `or_()`, `not_()`: 


In [ ]:
from sqlalchemy import not_

# comment traduisez vous cette requête ?
with app.app_context():
    query = db.select(Author).filter(
        Author.iconography.any(Iconography.date_lower <= 1900),
        not_(
            Author.author_name ==  "Atget Eugène, photographe"
        )
    )
    results = db.session.execute(query).scalars().all()
    print(results)


---

# Vers une application complète 

Tadada, on ajoute une page pour author, theme, place avec des renvois.